Import Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

Load Dataset

In [2]:
df = pd.read_csv(
    "../dataset/student_placement_prediction_dataset_2026.csv"
)

print("Dataset Shape:", df.shape)

print("\nDataset Columns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

Dataset Shape: (100000, 26)

Dataset Columns:
['student_id', 'age', 'gender', 'cgpa', 'branch', 'college_tier', 'internships_count', 'projects_count', 'certifications_count', 'coding_skill_score', 'aptitude_score', 'communication_skill_score', 'logical_reasoning_score', 'hackathons_participated', 'github_repos', 'linkedin_connections', 'mock_interview_score', 'attendance_percentage', 'backlogs', 'extracurricular_score', 'leadership_score', 'volunteer_experience', 'sleep_hours', 'study_hours_per_day', 'placement_status', 'salary_package_lpa']

First 5 Rows:


,student_id,age,gender,cgpa,branch,college_tier,internships_count,projects_count,certifications_count,coding_skill_score,...,mock_interview_score,attendance_percentage,backlogs,extracurricular_score,leadership_score,volunteer_experience,sleep_hours,study_hours_per_day,placement_status,salary_package_lpa
0,1,24,Male,7.53,IT,Tier 2,4,6,1,99.238568,...,72.647009,77.463863,2,63.382726,52.938240,Yes,6.7,3.6,Not Placed,0.00
1,2,21,Male,7.92,CSE,Tier 2,1,3,6,80.966123,...,61.699110,88.887600,1,73.694605,60.198856,No,4.4,2.3,Not Placed,0.00
2,3,22,Female,8.60,EEE,Tier 1,0,1,1,49.177184,...,87.396911,74.153265,0,63.329294,43.708803,No,8.8,5.9,Placed,11.99
3,4,24,Male,6.68,CSE,Tier 1,0,2,2,79.359084,...,58.401069,87.635955,1,47.636099,56.549154,Yes,8.1,4.4,Not Placed,0.00
4,5,20,Female,8.43,IT,Tier 3,1,4,3,65.018573,...,74.489201,79.120749,1,0.000000,67.268893,No,8.7,3.4,Placed,12.16


Check Target Distribution

In [3]:
print("Placement Status Distribution:")
print(df["placement_status"].value_counts())

print("\nPlacement Status Percentage:")
print(
    df["placement_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Placement Status Distribution:
placement_status
Placed        54459
Not Placed    45541
Name: count, dtype: int64

Placement Status Percentage:
placement_status
Placed        54.46
Not Placed    45.54
Name: proportion, dtype: float64


Prepare Data

In [4]:
df_model = df.drop(
    columns=[
        "student_id",
        "salary_package_lpa"
    ]
).copy()

df_model["placement_status"] = df_model["placement_status"].map({
    "Placed": 1,
    "Not Placed": 0
})

X = df_model.drop(
    columns=["placement_status"]
)

y = df_model["placement_status"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (100000, 23)
Target Shape: (100000,)


Load Trained Model

In [6]:
with open(
    "student_placement_model.pkl",
    "rb"
) as file:

    pipeline = pickle.load(file)

print("Trained model loaded successfully!")

Trained model loaded successfully!


Create Test Dataset

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

Training Samples: 80000
Testing Samples: 20000


Generate Predictions

In [9]:
y_pred = pipeline.predict(
    X_test
)

y_probability = pipeline.predict_proba(
    X_test
)[:, 1]

print("Predictions generated successfully!")

Predictions generated successfully!


Accuracy

In [10]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Model Accuracy:")
print(f"{accuracy * 100:.2f}%")

Model Accuracy:
55.57%


ROC-AUC

In [11]:
roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("ROC-AUC Score:")
print(f"{roc_auc:.4f}")

ROC-AUC Score:
0.5772


Classification Report

In [12]:
print("Classification Report")
print("=" * 60)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Not Placed",
            "Placed"
        ]
    )
)

Classification Report
              precision    recall  f1-score   support

  Not Placed       0.51      0.50      0.51      9108
      Placed       0.59      0.60      0.60     10892

    accuracy                           0.56     20000
   macro avg       0.55      0.55      0.55     20000
weighted avg       0.55      0.56      0.56     20000



Confusion Matrix

In [13]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[4567 4541]
 [4345 6547]]


Feature Importance

In [14]:
trained_model = pipeline.named_steps["model"]

trained_preprocessor = pipeline.named_steps[
    "preprocessor"
]

In [15]:
feature_names = (
    trained_preprocessor
    .get_feature_names_out()
)

In [16]:
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": trained_model.feature_importances_
})

In [17]:
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

In [18]:
print("Top 20 Important Features:")
display(
    feature_importance.head(20)
)

Top 20 Important Features:


,Feature,Importance
5,numerical__coding_skill_score,0.076820
12,numerical__mock_interview_score,0.072324
6,numerical__aptitude_score,0.066577
8,numerical__logical_reasoning_score,0.066411
7,numerical__communication_skill_score,0.064025
16,numerical__leadership_score,0.062106
15,numerical__extracurricular_score,0.060035
1,numerical__cgpa,0.056743
11,numerical__linkedin_connections,0.056456
13,numerical__attendance_percentage,0.055100


Final Evaluation

In [19]:
print("=" * 60)
print("MODEL EVALUATION SUMMARY")
print("=" * 60)

print(f"Dataset Records : {len(df):,}")
print(f"Total Features  : {len(X.columns)}")
print(f"Training Records: {len(X_train):,}")
print(f"Testing Records : {len(X_test):,}")

print(f"\nAccuracy : {accuracy * 100:.2f}%")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nEvaluation completed successfully.")

MODEL EVALUATION SUMMARY
Dataset Records : 100,000
Total Features  : 23
Training Records: 80,000
Testing Records : 20,000

Accuracy : 55.57%
ROC-AUC  : 0.5772

Evaluation completed successfully.
